In [2]:
import numpy as np
import cv2
import copy

def generate_field(wind_x, wind_y, shift):
    field = np.zeros((wind_y, wind_x, 3), np.uint8)
    field[:, :, 0] = 12
    field[:, :, 1] = 134
    field[:, :, 2] = 73

    points = [(np.random.randint(wind_x - 4 * shift) + shift,
               np.random.randint(wind_y - 2 * shift) + shift) for i in range(20)]

    mask = np.zeros_like(field[:, :, 0])

    for i in range(len(points) - 1):
        cv2.line(mask, points[i], points[i + 1], 255, 200)
    
    #cv2.imshow("mask", mask)

    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((143, 143)))
    
    #cv2.imshow("closed", mask)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    upper_left_ind = 0
    min_sum = 1000000
    
    for i in range(len(contours[0])):
        if (contours[0][i][0][0] + contours[0][i][0][1] < min_sum):
            min_sum = contours[0][i][0][0] + contours[0][i][0][1]
            upper_left_ind = i
    
    path_with_start = [(shift, shift // 2)]
    
    for i in range(len(contours[0])):
        path_with_start.append(contours[0][(i + upper_left_ind) % len(contours[0])][0])
    
    for i in range(len(path_with_start) - 250):
        cv2.line(field, path_with_start[i], path_with_start[i + 1], (12, 123, 234), 5)
    
    return field

def get_v_adaptive_mask(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    otsu_th, mask = cv2.threshold(hsv[:, :, 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return mask

def load_templates(template_paths, template_size):
    templates = {}
    
    for template_path in template_paths:
        template_img_ = cv2.imread(template_path)

        assert template_img_ is not None, "Cannot find image " + str(name)

        #workaround for data with bboxes :/
        template_img = template_img_[1 : -1, 1 : -1, :]

        template_img = cv2.resize(template_img, (template_size, template_size))

        template_name = template_path.split("_")[0]
        
        #print(template_name)
        
        templates.update({template_name : {"0" : template_img}})

        for i in range(1, 4):
            template_img = cv2.rotate(template_img, cv2.ROTATE_90_CLOCKWISE)
            templates[template_name].update({str(90*i) : template_img})
    
    return templates

def place_sign(field, templates, template_size, x, y, template_name, rotation, offset = 10):
    field[y - template_size // 2 - offset: y + template_size // 2 + offset,
          x - offset: x + template_size + offset, :] = 255
    
    field[y - template_size // 2: y + template_size // 2,
          x : x + template_size, :] = templates[template_name][rotation]

def generate_constant_field(wind_x, wind_y, templates, template_size):
    field = np.zeros((wind_y, wind_x, 3), np.uint8)
    field[:, :, 0] = 12
    field[:, :, 1] = 134
    field[:, :, 2] = 73
    
    segments = [[(400, 250), (500, 400), (720, 400)]]
    
    place_sign(field, templates, template_size, 820, 400, "up", "90")
    place_sign(field, templates, template_size, 1220, 400, "right", "90")
    place_sign(field, templates, template_size, 1220, 800, "up", "180")
    place_sign(field, templates, template_size, 1220, 1200, "left", "180")
    
    #field[400 - template_size // 2: 400 + template_size // 2,
    #      520 : 520 + template_size, :] = templates["up"]["90"]

#     field[400 - template_size // 2: 400 + template_size // 2,
#           820 : 820 + template_size, :] = templates["right"]["90"]

#     field[700 - template_size // 2: 700 + template_size // 2,
#           820 : 820 + template_size, :] = templates["up"]["180"]

#     field[900 - template_size // 2: 900 + template_size // 2,
#           820 : 820 + template_size, :] = templates["left"]["180"]

    for i in range(len(segments)):
        for j in range(len(segments[i]) - 1):
            cv2.line(field, segments[i][j], segments[i][j + 1], (12, 123, 234), 5)
    
    return field

template_size = 50

templates = load_templates(["left_template.png", "right_template.png", "up_template.png"], template_size)

#field = generate_constant_field(1900, 1400, templates, template_size)
field = generate_field(1900, 1400, 200)

cv2.imshow("field", field)
cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.waitKey(10)

-1

In [4]:
import numpy as np
import cv2
import copy
import math
from scipy.special import comb

class Controller:
    def __init__(self, max_v_linear, TEMPL_SZ, templates):
        self.max_v_linear = max_v_linear
        
        self.last_feedback = (0, 0, 0)
        self.last_spline = []
        
        self.TEMPL_SZ = TEMPL_SZ
        
        self.templates = templates
    
    def calc_IoU(self, mask1, mask2):
        intersection = cv2.bitwise_and(mask1, mask2).sum()
        union = cv2.bitwise_or(mask1, mask2).sum()
        
        return intersection / union
    
    def get_v_adaptive_mask(self, image):
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        
        otsu_th, mask = cv2.threshold(hsv[:, :, 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        return mask
    
    def get_mask(self, image, thresholds, la = 10, ha = 1000000):
        mask = cv2.inRange(image, thresholds[0], thresholds[1])
        
        cv2.imshow("mask", mask)
        #cv2.waitKey(0)
            
        connectivity = 4
        output = cv2.connectedComponentsWithStats(mask, connectivity, cv2.CV_32S)
        
        num_labels = output[0]
        labels = output[1]
        stats = output[2]
        
        filtered = np.zeros_like(mask)
        
        for i in range(1, num_labels):
            a = stats[i, cv2.CC_STAT_AREA]
            t = stats[i, cv2.CC_STAT_TOP]
            l = stats[i, cv2.CC_STAT_LEFT]
            w = stats[i, cv2.CC_STAT_WIDTH]
            h = stats[i, cv2.CC_STAT_HEIGHT]

            if (a >= la and a <= ha):
                filtered[np.where(labels == i)] = 255
        
        return filtered

    #https://stackoverflow.com/questions/12643079/bézier-curve-fitting-with-scipy
    def bernstein_poly(self, i, n, t):
        """
        The Bernstein polynomial of n, i as a function of t
        """

        return comb(n, i) * ( t**(n-i) ) * (1 - t)**i

    def bezier_curve(self, points, nTimes=1000):
        """
        Given a set of control points, return the
        bezier curve defined by the control points.

        points should be a list of lists, or list of tuples
        such as [ [1,1], 
                    [2,3], 
                    [4,5], ..[Xn, Yn] ]
            nTimes is the number of time steps, defaults to 1000

            See http://processingjs.nihongoresources.com/bezierinfo/
        """

        nPoints = len(points)
        xPoints = np.array([p[0] for p in points])
        yPoints = np.array([p[1] for p in points])

        t = np.linspace(0.0, 1.0, nTimes)

        polynomial_array = np.array([ self.bernstein_poly(i, nPoints-1, t) for i in range(0, nPoints)   ])

        xvals = np.dot(xPoints, polynomial_array)
        yvals = np.dot(yPoints, polynomial_array)

        return xvals, yvals

    def weighted_point_distance(self, p, lowest):
        return math.sqrt((lowest[0, 0] - p[0, 0])**2 + 2 * (lowest[0, 1] - p[0, 1])**2)

    def get_spline(self, image, mask):
        #COMMENTED
        #cv2.imwrite("mask.png", mask)
        
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

        filtered = []
        
        if (len(contours) == 0):
            #print("zhopa")
            return [], [], [[0, 0]], [[0, 0]]
        
        contour = contours[0]
        lowest_point_y = 0

        for cnt in contours:
            lowest = sorted(cnt, key = lambda p : p[0, 1])[-1]
            
            if (lowest[0, 1] > lowest_point_y):
                lowest_point_y = lowest[0, 1]
                
                l = cv2.arcLength(cnt, True)
                approx = cv2.approxPolyDP(cnt, l * 0.05, True)
                contour = approx
            
        if (lowest_point_y == 0):
            return [], [], [[0, 0]]
        
        sorted_contour = sorted(contour, key = lambda p : p[0, 1])
        lowest = sorted_contour[-1]
        
        most_distant = sorted(contour, key = lambda p : self.weighted_point_distance(p, lowest))[-1]
        cv2.circle(image, lowest[0], 25, (100, 100, 100), 4)
        #cv2.circle(image, most_distant[0], 10, (23, 234, 212), 4)
        
        point_sequence = []
        
        lowest_index = 0
        
        for i in range(len(contour)):
            if (np.array_equal(lowest, contour[i])):
                lowest_index = i
                break
            
        for i in range(lowest_index, lowest_index + len(contour)):
            curr = contour[i % len(contour)]
                            
            point_sequence.append(curr[0])
            
            #cv2.circle(image, curr[0], 10, (223, 234, int(255.0 / len(contour) * i)), 5)
            
            if (np.array_equal(curr, most_distant)):
                break
        
        densified_sequence = []
        densification_coeff = 3
        
        for i in range(len(point_sequence) - 1):
            for j in range(densification_coeff):
                coeff = float(j) / densification_coeff
                new_point = point_sequence[i] * (1 - coeff) + point_sequence[i + 1] * coeff
                
                densified_sequence.append(new_point)

            densified_sequence.append(point_sequence[i])
        
        densified_sequence.append(point_sequence[-1])
        
        #for point in densified_sequence:
        #    cv2.circle(image, (int(point[0]), int(point[1])), 8, (43, 32, 233), -1)
        
        xvals, yvals = self.bezier_curve(densified_sequence, nTimes=10)

        #for i in range(len(xvals)):
        #    cv2.circle(image, (int(xvals[i]), int(yvals[i])), 16, (234, 234, 23), -1)
        
        #cv2.drawContours(image, [contour], -1, (0,255,0), 3)

        cv2.imshow("output", image)
        cv2.waitKey(1)
        
        return xvals, yvals, lowest, most_distant
    
    def cut_sign(self, image):
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        
        mask = cv2.inRange(hsv, (0, 0, 240), (255, 255, 255))
        
        #print("acadcacdadcasdc")
        cv2.imshow("mask", mask)
        
        output = cv2.connectedComponentsWithStats(mask, 4, cv2.CV_32S)
        num_labels = output[0]
        labels = output[1]
        stats = output[2]
        
        if (num_labels < 2):
            return None, 0, 0, 0
        
        biggest_component_num = 1
        max_area = 10

        for i in range(1, num_labels):
            area = stats[i, cv2.CC_STAT_AREA]
            
            if (max_area < area):
                max_area = area
                biggest_component_num = i
            
        mask_for_biggest = np.zeros_like(mask)
        
        mask_for_biggest[np.where(labels == biggest_component_num)] = 255
        
        contours, _ = cv2.findContours(mask_for_biggest, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        rect = cv2.minAreaRect(contours[0])
        box = cv2.boxPoints(rect)
        
        angle = math.atan((box[0, 1] - box[1, 1]) / (box[0, 0] - box[1, 0]))
        
        top    = stats[biggest_component_num, cv2.CC_STAT_TOP]
        left   = stats[biggest_component_num, cv2.CC_STAT_LEFT]
        width  = stats[biggest_component_num, cv2.CC_STAT_WIDTH]
        height = stats[biggest_component_num, cv2.CC_STAT_HEIGHT]
        
        cX = int(width // 2)
        cY = int(height // 2)
        
        crop_x = left + width // 2
        crop_y = top + height // 2

        crop = image[top: top + height, left: left + width]
        #cv2.imshow("crop", crop)
        
        if (math.isnan(angle)):
            return None, 0, 0, 0
        
        #print("angle", angle)
        
        M = cv2.getRotationMatrix2D((cX, cY), int(np.degrees(angle)), 1.0)
        rotated = cv2.warpAffine(crop, M, (width, height))
        
        def rotate(origin, point, angle):
            ox, oy = origin
            px, py = point

            qx = ox + math.cos(angle) * (px - ox) - math.sin(angle) * (py - oy)
            qy = oy + math.sin(angle) * (px - ox) + math.cos(angle) * (py - oy)
            
            return (int(qx), int(qy))
        
        rotated_points = []
        
        for point in box:
            point_in_crop = (point[0] - left, point[1] - top)
            
            rotated_point = rotate((cX, cY), point_in_crop, -angle)
            rotated_points.append(rotated_point)
        
        marker_mask = np.zeros_like(rotated[:, :, 0])
        #cv2.fillPoly(marker_mask, pts = rotated_points, color=(255))
        
        #print(box)
        #print(rotated_points)
        
        #for i, point in enumerate(rotated_points):
        #    cv2.putText(rotated, str(i), (int(point[0]), int(point[1])), cv2.FONT_HERSHEY_SIMPLEX, 
        #                       1, (255, 0, 0), 2, cv2.LINE_AA)
        
        box = np.int32(rotated_points)#(box)
        #cv2.drawContours(rotated, [box], 0, (0,0,255), 2)
        cv2.drawContours(marker_mask, [box], 0, (255), -1)
        
        #cv2.imshow("rotated", rotated)
        #cv2.imshow("marker_mask", marker_mask)
        
        cropped_marker = cv2.bitwise_and(rotated, rotated, mask = marker_mask)
        #cv2.imshow("cropped marker", cropped_marker)
        
        hsv_marker = cv2.cvtColor(cropped_marker, cv2.COLOR_BGR2HSV)
        #marker_mask = cv2.inRange(hsv_marker, (0, 0, 0), (255, 255, 105))
        
        #blur = cv.GaussianBlur(img,(5,5),0)
        otsu_th, marker_mask = cv2.threshold(hsv_marker[:, :, 2], 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        
        #print("otsu th", otsu_th)
        
        output = cv2.connectedComponentsWithStats(marker_mask, 4, cv2.CV_32S)
        num_labels = output[0]
        labels = output[1]
        stats = output[2]
        
        if (num_labels < 2):
            return None, 0, 0, 0
        
        densest_component_num = 1
        max_density = 0.01

        for i in range(1, num_labels):
            area   = stats[i, cv2.CC_STAT_AREA]
            width  = stats[i, cv2.CC_STAT_WIDTH]
            height = stats[i, cv2.CC_STAT_HEIGHT]
            
            curr_density = area / (width * height)
            
            if (max_density < curr_density and area > 100):
                max_density = curr_density
                densest_component_num = i

        top    = stats[densest_component_num, cv2.CC_STAT_TOP]
        left   = stats[densest_component_num, cv2.CC_STAT_LEFT]
        width  = stats[densest_component_num, cv2.CC_STAT_WIDTH]
        height = stats[densest_component_num, cv2.CC_STAT_HEIGHT]

        sign_crop = cropped_marker[top: top + height, left: left + width]
        
        cv2.imshow("sign crop", sign_crop)
        
        #dist = plot_dist(hsv_marker[:, :, 2])
        #print(dist.shape)
        #cv2.imshow("dist", dist[:, :, 1])
        
        cv2.imshow("otsu", marker_mask)
                
        return sign_crop, angle, crop_x, crop_y
    
    def classify_sign(self, crop):
        resized = cv2.resize(crop, (self.TEMPL_SZ, self.TEMPL_SZ))
        
        mask = self.get_v_adaptive_mask(resized)
        
        best_template = "aboba"
        best_angle = "0"
        max_IoU = -1.0
        
        for template_name, template in self.templates.items():
            for angle, rotated_template in template.items():
                #TODO: obtain and store rotated template masks somewhere to avoid recalculation
                rotated_template_mask = self.get_v_adaptive_mask(rotated_template)
                
                IoU = self.calc_IoU(mask, rotated_template_mask)
                
                #print(IoU)
                
                if (IoU > max_IoU):
                    max_IoU = IoU
                    best_template = template_name
                    best_angle = angle
        
        return best_template, best_angle, max_IoU

    def generate_trajectory(self, frame, cutoff = 130):
        crop, angle, crop_x, crop_y = self.cut_sign(frame)
        
        h, w, _ = frame.shape

        x, y = w // 2, 0
        lowest = [(w // 2, 0)]
        xvals, yvals = [], []
        
        #print(crop)
        
        if (crop is not None):
            #print(crop.shape)
            cv2.imshow("crop", crop)

        if ((crop is not None) and min(crop.shape[:2]) > 20):
            cv2.imshow("crop", crop)
            #print("sign")
            
            sign_type, rotation, max_IoU = self.classify_sign(crop)
            
            print(sign_type, rotation, max_IoU)
            
            if (max_IoU > 0.4):
                cv2.putText(frame, "sign", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 
                           1, (255, 0, 0), 2, cv2.LINE_AA)
                
                if (h - crop_y < 150):
                    sign_angle = 0

                    if (sign_type == "right"):
                        sign_angle = np.pi / 2

                    if (sign_type == "left"):
                        sign_angle = -np.pi / 2

                    full_angle = angle + int(rotation) / 180 * np.pi + sign_angle

                    #print("full angle", full_angle)
                    #TODO: calculate desired position - 1 m from the arrow if arrow is close,
                    #arrow itself if it is far away

                    tx = 50 * np.sin(full_angle)
                    ty = 50 * np.cos(full_angle)

                    cv2.line(frame, (crop_x, crop_y), (crop_x + int(tx), crop_y - int(ty)), (123, 123, 234), 4)

                    #print("tx", tx)

                    return tx + w // 2, ty, lowest, xvals, yvals
                
                else:
                    return crop_x, crop_y, lowest, xvals, yvals

        if (crop is None or ((crop is not None) and min(crop.shape[:2])) <= 50):
            #print("line")
            
            threshold = [(11, 122, 233), (13, 124, 235)]
        
            mask = self.get_mask(frame, threshold, 40)
            
            xvals, yvals, lowest, most_distant = self.get_spline(frame, mask)
                        
            if (math.sqrt((lowest[0][1] - most_distant[0][1])**2 +
                          (lowest[0][0] - most_distant[0][0])**2) > 20):
                cv2.putText(frame, "line", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 
                               1, (255, 0, 0), 2, cv2.LINE_AA)

                for i in range(len(xvals)):
                    if (yvals[i] < cutoff):
                        x = xvals[i]
                        y = yvals[i]        
                        break
            
                return x, y, lowest, xvals, yvals
        
        cv2.putText(frame, "odometry", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 
                           1, (255, 0, 0), 2, cv2.LINE_AA)

        x, y = w // 2, 0
        lowest = [(w // 2, 0)]
        
        return x, y, lowest, xvals, yvals
    
    def generate_feedback(self, observation, last_state):
        vx, vy, omega = 0.1, 0.1, 0.01
        
        feedback = (vx, vy, omega)
        
        self.last_feedback = feedback
        
        return feedback

    def draw(self, field_image, observation, PoV_size = 300):
        for i in range(len(self.last_spline[0])):
            cv2.circle(observation, (int(self.last_spline[0][i]), int(self.last_spline[1][i])),
                       6, (240, 240, 240), 2)
        
        resized_pov = cv2.resize(observation, (PoV_size, PoV_size)) - 30
        
        field_image[:PoV_size, -PoV_size:, :] = resized_pov

class Controller_heuristic(Controller):
    def __init__(self, max_v_linear, TEMPL_SIZE, templates):
        super().__init__(max_v_linear, TEMPL_SIZE, templates)

    def generate_feedback(self, observation, last_state):
        #vx, vy, omega = 0.1, 0.1, 0.01
        x, y, theta = last_state
        
        x_next, y_next, lowest, xvals, yvals = self.generate_trajectory(observation)
        
        vx = self.max_v_linear
        
        x_lowest = lowest[0][0]
        lateral_compensation = 0.3 * (x_lowest - observation.shape[1] // 2)
        vy = lateral_compensation
        
        #print("next - lowest", x_next, x_lowest)
        
        omega = 0.05 * np.sign(x_next - x_lowest)
        
        feedback = (vx, vy, omega)
        
        self.last_feedback = feedback
        self.last_spline = (xvals, yvals)
        
        #print("feedback", feedback)
        
        return feedback

class Robot:
    def __init__(self, x0, y0, theta0, dt, max_v_linear, max_omega):
        self.x = x0
        self.y = y0
        self.theta = theta0
        self.dt = dt
        self.max_v_linear = max_v_linear
        self.max_omega = max_omega
    
    def move_global(self, control):
        vx, vy, omega = control
        
        omega = np.clip(omega, -self.max_omega, self.max_omega)
        v_linear = np.sqrt(vx**2 + vy**2)
        
        if (v_linear > self.max_v_linear):
            vx = vx / v_linear * self.max_v_linear
            vy = vy / v_linear * self.max_v_linear
        
        self.x += vx * self.dt
        self.y += vy * self.dt
        self.theta += omega * self.dt

    def move_local(self, control):
        vx, vy, omega = control
        
        vx_rot = (vx * np.cos(self.theta) - vy * np.sin(self.theta)) * (1 + np.random.randn() * 0)
        vy_rot = (vx * np.sin(self.theta) + vy * np.cos(self.theta)) * (1 + np.random.randn() * 0)
        
        self.move_global((vx_rot, vy_rot, omega))
    
    def draw(self, canvas):
        cv2.circle(canvas, (int(self.x), int(self.y)), 25, (123, 234, 234), 2)
        cv2.line(canvas, (int(self.x), int(self.y)),
                 (int(self.x + 40 * np.cos(self.theta)), int(self.y + 40 * np.sin(self.theta))),
                 (253, 223, 34), 5)
    
    def get_state(self):
        return (self.x, self.y, self.theta)

class Simulator:
    def __init__(self, visible_zone_size, FIELD_X, FIELD_Y, shift, TEMPLATE_SIZE, templates):
        self.visible_zone_size = visible_zone_size
        self.last_observation = np.zeros((self.visible_zone_size,
                                          self.visible_zone_size, 3), np.uint8)
        
        self.FIELD_X = FIELD_X
        self.FIELD_Y = FIELD_Y
        
        self.TEMPLATE_SIZE = TEMPLATE_SIZE
        
        templates = templates

        self.field = generate_constant_field(self.FIELD_X, self.FIELD_Y, templates, self.TEMPLATE_SIZE)
        
        self.M = np.eye(2)
        self.last_state = (100, 100, 0)
        
        self.observation_contour = [(0,                      -self.visible_zone_size // 2),
                                    (self.visible_zone_size, -self.visible_zone_size // 2),
                                    (self.visible_zone_size,  self.visible_zone_size // 2),
                                    (0,                       self.visible_zone_size // 2)]
    
    def get_observation(self, state):
        self.last_state = state
        x, y, theta = self.last_state
        
        self.M = cv2.getRotationMatrix2D((x, y), int(np.degrees(theta)), 1.0)
                
        rotated = cv2.warpAffine(self.field, self.M, (self.FIELD_X, self.FIELD_Y))
        
        #cv2.imshow("rotated", rotated)
        
        crop = rotated[int(y - self.visible_zone_size // 2) : int(y + self.visible_zone_size // 2),
                       int(x) : int(x + self.visible_zone_size), :]
        
        crop = cv2.rotate(crop, cv2.ROTATE_90_COUNTERCLOCKWISE)
        
        return crop
    
    def rotate_and_move_contour(self, contour, shift_and_rotation):
        xc, yc, theta = shift_and_rotation
        
        rotated_contour = []
        
        for el in contour:
            x, y = el
            
            xr = int(x * np.cos(theta) - y * np.sin(theta) + xc)
            yr = int(x * np.sin(theta) + y * np.cos(theta) + yc)
            
            rotated_contour.append([xr, yr])
        
        return rotated_contour
    
    def draw_observation(self, canvas):
        current_observation_contour = self.rotate_and_move_contour(self.observation_contour,
                                        self.last_state)
        
        cv2.drawContours(canvas, [np.intp(current_observation_contour)], 0, (123, 234, 234), 2)
    
    def get_field_image(self):
        return copy.deepcopy(self.field)

shift = 230
visible_zone_size = 300
FIELD_X, FIELD_Y = 2400, 1500

max_v_linear = 3

template_size = 50

templates = load_templates(["left_template.png", "right_template.png", "up_template.png"],
                           template_size)

simulator = Simulator(visible_zone_size, FIELD_X, FIELD_Y, shift, template_size, templates)

controller = Controller_heuristic(max_v_linear = max_v_linear, TEMPL_SIZE = template_size,
                                  templates = templates)
#controller = Controller_MPC(max_v_linear = max_v_linear)

#robot = Robot(450, 400, 0, dt = 2.0, max_v_linear = max_v_linear, max_omega = 0.05)
robot = Robot(250, 250, 0.5, dt = 2.0, max_v_linear = max_v_linear, max_omega = 0.05)

running = True

while(running):
    state = robot.get_state()
    observation = simulator.get_observation(state)
        
    control = controller.generate_feedback(observation, state)
    robot.move_local(control)
    
    field_image = simulator.get_field_image()
    robot.draw(field_image)
    
    simulator.draw_observation(field_image)
    controller.draw(field_image, observation, PoV_size = 500)
    
    cv2.imshow("marathon", field_image)
        
    key = cv2.waitKey(150)
    
    if (key == ord('q')):
        break

cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.waitKey(100)

-1

In [ ]:
import numpy as np
import cv2
from enum import Enum
import math

#Generates limited proportional feedback
class Feedback:
    def __init__(self, target_value, tolerance, max_velocity, gain):
        self.target_value = target_value
        self.tolerance    = tolerance
        self.max_velocity = max_velocity
        self.gain         = gain
    
    def generate_feedback(self, value):
        feedback = (value - self.target_value) * self.gain
        
        feedback = np.clip(feedback, -self.max_velocity, self.max_velocity)
        
        return feedback
    
    def stabilized(self, value):
        return abs(value - self.target_value) < self.tolerance

class Approach:
    def __init__(self):
        pass

    def generate_action(self, distance, angle):
        return 0, 0, 0

    def criteria_is_met(self):
        return True

#This approach performs open-loop motion until odometry reaches target for all 3 coordinates
class Approach_by_odometry(Approach):
    def __init__(self, target_x, target_y, target_a,
                       distance_tolerance, angle_tolerance,
                       max_x_velocity, max_y_velocity, max_rotary_velocity,
                       x_velocity, y_velocity, rotary_velocity):
        super().__init__()
        
        self.curr_x = 0
        self.curr_y = 0
        self.curr_a = 0

        self.target_x = target_x
        self.target_y = target_y
        self.target_a = target_a

        self.distance_tolerance = distance_tolerance
        self.angle_tolerance    = angle_tolerance

        self.max_x_velocity      = max_x_velocity
        self.max_y_velocity      = max_y_velocity
        self.max_rotary_velocity = max_rotary_velocity

        self.x_velocity = x_velocity
        self.y_velocity = y_velocity
        self.rotary_velocity = rotary_velocity
        
    def restart(self):
        self.curr_x = 0
        self.curr_y = 0
        self.curr_a = 0

    def generate_action(self, distance, angle):
        vx = self.x_velocity
        vy = self.y_velocity
        omega = self.rotary_velocity

        self.curr_x += vx
        self.curr_y += vy
        self.curr_a += omega
        
        return vx, vy, omega
    
    def criteria_is_met(self, distance, angle):
        #print(abs(self.target_x - self.curr_x),
        #      abs(self.target_y - self.curr_y),
        #      abs(self.target_a       - angle))

        return (abs(self.target_x - self.curr_x) < self.distance_tolerance and
                abs(self.target_y - self.curr_y) < self.distance_tolerance and
                abs(self.target_a       - angle) < self.angle_tolerance)

#This approach works by forward motion and rotation only. No lateral motion is used
class Approach_by_distance_and_heading(Approach):
    def __init__(self, target_distance, target_angle,
                       distance_tolerance, angle_tolerance,
                       max_linear_velocity, max_rotary_velocity,
                       forward_gain, rotary_gain):
        super().__init__()
        
        self.target_distance = target_distance
        self.target_angle    = target_angle
        
        self.distance_tolerance = distance_tolerance
        self.angle_tolerance    = angle_tolerance
        
        self.max_linear_velocity = max_linear_velocity
        self.max_rotary_velocity = max_rotary_velocity
        
        self.forward_gain = forward_gain
        self.rotary_gain  = rotary_gain
        
        self.distance_feedback = Feedback(self.target_distance, self.distance_tolerance,
                                          self.max_linear_velocity, self.forward_gain)

        self.angle_feedback = Feedback(self.target_angle, self.angle_tolerance,
                                          self.max_rotary_velocity, self.rotary_gain)
        
    def generate_action(self, distance, angle):
        vx = self.distance_feedback.generate_feedback(distance)
        vy = 0
        omega = self.angle_feedback.generate_feedback(angle)
        
        return vx, vy, omega
    
    def criteria_is_met(self, distance, angle):
        return (abs(self.target_distance - distance) < self.distance_tolerance and
                abs(self.target_angle    - angle)    < self.angle_tolerance)

class States(Enum):
    INIT = 0
    SEARCHING_FOR_BALL = 1
    APPROACHING_BALL = 2
    TAKING_BALL = 3
    ESCAPING_BALL_POST = 4
    SEARCHING_FOR_BASKET = 5
    APPROACHING_SHOOTING_POSITION = 6
    THROWING_BALL = 7
    EXIT = 8

class Actions(Enum):
    WALKING = 0
    TAKING_BALL = 1
    THROWING_BALL = 2

class Basketball_controller:
    def __init__(self, target_distance_to_ball, target_angle_to_ball,
                 target_distance_to_basket, target_angle_to_basket,
                distance_tolerance, angle_tolerance, max_forward_velocity,
                max_lateral_velocity, max_rotary_velocity,
                forward_gain, rotary_gain):
        self.state = States.INIT
        self.prev_state = self.state
        
        self.approach_ball = Approach_by_distance_and_heading(target_distance_to_ball, target_angle_to_ball,
            distance_tolerance, angle_tolerance, max_forward_velocity, max_rotary_velocity,
            forward_gain, rotary_gain)

        self.search_ball = Approach_by_odometry(0, 0, 4 * np.pi, distance_tolerance, angle_tolerance,
            max_forward_velocity, max_lateral_velocity, max_rotary_velocity, 0, 0, max_rotary_velocity)

        self.escape_post = Approach_by_odometry(0, 0.34, 0, distance_tolerance, angle_tolerance,
            max_forward_velocity, max_lateral_velocity, max_rotary_velocity, 0, max_lateral_velocity, 0)

        self.approach_basket = Approach_by_distance_and_heading(target_distance_to_basket, target_angle_to_basket,
            distance_tolerance, angle_tolerance, max_forward_velocity, max_rotary_velocity,
            forward_gain, rotary_gain)
        
        self.search_basket = Approach_by_odometry(0, 0, 4 * np.pi, distance_tolerance, angle_tolerance,
            max_forward_velocity, max_lateral_velocity, max_rotary_velocity, 0, 0, max_rotary_velocity)
    
    def generate_action(self, observation):
        #separate into two functions: handling FSM and generating action itself

        (x_ball_local, y_ball_local, ball_visible), (x_basket_local, y_basket_local, basket_visible) = observation

        self.prev_state = self.state

        if (self.state == States.INIT):
            self.state = States.SEARCHING_FOR_BALL
        
        if (self.state == States.SEARCHING_FOR_BALL):
            if (ball_visible):
                self.state = States.APPROACHING_BALL

        if (self.state == States.APPROACHING_BALL):
            if (not ball_visible):
                self.state = States.SEARCHING_FOR_BALL
            
            else:
                distance_to_ball = np.sqrt(x_ball_local**2 + y_ball_local**2)
                angle_to_ball = math.atan2(y_ball_local, x_ball_local)

                ball_approached = self.approach_ball.criteria_is_met(distance_to_ball, angle_to_ball)

                if (ball_approached):
                    self.state = States.TAKING_BALL

        if (self.state == States.TAKING_BALL):
            self.state = States.ESCAPING_BALL_POST

            #UNA PREGUNTA!! (VOPROS)
            #How to understand if the ball is taken, given that it is a prolongued process?
            #if (ball_taken):
            #   self.state = States.SEARCHING_FOR_BASKET

        if (self.state == States.ESCAPING_BALL_POST):
            if (self.escape_post.criteria_is_met(0, 0)):
                #print("khe")
                self.state = States.SEARCHING_FOR_BASKET

        if (self.state == States.SEARCHING_FOR_BASKET):
            if (basket_visible):
                self.state = States.APPROACHING_SHOOTING_POSITION

        if (self.state == States.APPROACHING_SHOOTING_POSITION):
            if (not basket_visible):
                self.state = States.SEARCHING_FOR_BASKET
            
            else:
                distance_to_basket = np.sqrt(x_basket_local**2 + y_basket_local**2)
                angle_to_basket = math.atan2(y_basket_local, x_basket_local)

                basket_approached = self.approach_basket.criteria_is_met(distance_to_basket, angle_to_basket)

                if (basket_approached):
                    self.state = States.THROWING_BALL

        if (self.state == States.THROWING_BALL):
            self.state = States.EXIT

            #same as for TAKING_BALL
            #if (ball_thrown):
            #   self.state = States.EXIT

        #generating action
        action_name = Actions.WALKING
        vx = 0
        vy = 0
        omega = 0
        
        if (self.state == States.SEARCHING_FOR_BALL):
            if (self.state != self.prev_state):
                self.search_ball.restart()
            
            vx, vy, omega = self.search_ball.generate_action(0, 0)

        if (self.state == States.APPROACHING_BALL):
            vx, vy, omega = self.approach_ball.generate_action(distance_to_ball, angle_to_ball)

        if (self.state == States.TAKING_BALL):
            #pregunta
            pass

        if (self.state == States.ESCAPING_BALL_POST):
            vx, vy, omega = self.escape_post.generate_action(0, 0)

            #print(vx, vy, omega)

        if (self.state == States.SEARCHING_FOR_BASKET):
            if (self.state != self.prev_state):
                self.search_basket.restart()
            
            vx, vy, omega = self.search_basket.generate_action(0, 0)

        if (self.state == States.APPROACHING_SHOOTING_POSITION):
            distance_to_basket = np.sqrt(x_basket_local**2 + y_basket_local**2)
            angle_to_basket = math.atan2(y_basket_local, x_basket_local)
            
            vx, vy, omega = self.approach_basket.generate_action(distance_to_basket, angle_to_basket)

        if (self.state == States.THROWING_BALL):
            self.state = States.EXIT

        if (self.state == States.EXIT):
            pass
            #print("Obrigado")

        return (vx, vy, omega), action_name, self.state
    
    def get_state(self):
        return self.state

class Robot:
    def __init__(self, x0, y0, a0, dt):
        self.x = x0
        self.y = y0
        self.a = a0 #alpha
        self.dt = dt
    
    def walk(self, velocities):
        vx, vy, omega = velocities

        self.x += vx * self.dt * math.cos(self.a) - vy * self.dt * math.sin(self.a)
        self.y += vx * self.dt * math.sin(self.a) + vy * self.dt * math.cos(self.a)
        self.a += omega * dt

    def get_state(self):
        return (self.x, self.y, self.a)
        
class Object:
    def __init__(self, x_obj, y_obj):
        self.x_obj = x_obj
        self.y_obj = y_obj
        
    def get_observation(self, x_robot, y_robot, a_robot, FoV):
        R = np.array([[np.cos(a_robot), np.sin(a_robot)],
                      [-np.sin(a_robot), np.cos(a_robot)]])
        
        X_object_glob = np.array([[self.x_obj], [self.y_obj]])
        X_robot_glob  = np.array([[x_robot], [y_robot]])

        translated_rotated = R @ (X_object_glob - X_robot_glob)

        x_loc, y_loc = translated_rotated[:, 0]

        angle = math.atan2(y_loc, x_loc)

        object_visible = abs(angle) < FoV / 2

        return x_loc, y_loc, object_visible

class World:
    def __init__(self, x_ball, y_ball, x_basket, y_basket):
        self.x_ball = x_ball
        self.y_ball = y_ball

        self.x_basket = x_basket
        self.y_basket = y_basket
        
        self.ball   = Object(self.x_ball, self.y_ball)
        self.basket = Object(self.x_basket, self.y_basket)
    
    def get_observation(self, robot_state, FoV):
        x_robot, y_robot, a_robot = robot_state

        x_ball_local, y_ball_local, ball_visible       = self.ball.get_observation(x_robot, y_robot, a_robot, FoV)
        x_basket_local, y_basket_local, basket_visible = self.basket.get_observation(x_robot, y_robot, a_robot, FoV)
        
        return (x_ball_local, y_ball_local, ball_visible), \
               (x_basket_local, y_basket_local, basket_visible)

class Simulator:
    def __init__(self, x_ball, y_ball, x_basket, y_basket, x_robot, y_robot, a_robot,
                 dt, max_simulation_steps, FoV):
        self.x_ball = x_ball
        self.y_ball = y_ball

        self.x_basket = x_basket
        self.y_basket = y_basket

        self.x_robot = x_robot
        self.y_robot = y_robot
        self.a_robot = a_robot

        self.dt = dt
        self.max_simulation_steps = max_simulation_steps

        self.FoV = FoV

        self.robot = Robot(self.x_robot, self.y_robot, self.a_robot, self.dt)

        target_distance_to_ball = 0.2
        target_angle_to_ball = 0
        target_distance_to_basket = 0.2
        target_angle_to_basket = 0
        distance_tolerance = 0.05
        angle_tolerance = 0.1
        max_forward_velocity = 0.05
        max_lateral_velocity = 0.05
        max_rotary_velocity = 0.3
        forward_gain = 1
        rotary_gain = 1

        self.controller = Basketball_controller(target_distance_to_ball, target_angle_to_ball,
            target_distance_to_basket, target_angle_to_basket,
            distance_tolerance, angle_tolerance, max_forward_velocity, max_lateral_velocity,
            max_rotary_velocity, forward_gain, rotary_gain)
        
        self.world = World(self.x_ball, self.y_ball, self.x_basket, self.y_basket)

        self.max_simulation_steps = max_simulation_steps
        self.iter_num = 0
        
        self.state_trajectory = []
        self.state_name_trajectory = []
        self.action_trajectory = []
    
    def run_simulation_step(self):
        robot_state = self.robot.get_state()
        
        observation = self.world.get_observation(robot_state, self.FoV)
        
        #(vx, vy, omega), action_name, self.state
        action, action_name, state_name = self.controller.generate_action(observation)

        self.robot.walk(action)
        
        self.state_trajectory.append(robot_state)
        self.action_trajectory.append(action)
        self.state_name_trajectory.append(state_name)

        self.iter_num += 1
        
    def get_trajectories(self):
        return self.state_trajectory, self.action_trajectory, self.state_name_trajectory
    
    def running(self):
        return (self.controller.get_state() != States.EXIT and self.iter_num < self.max_simulation_steps)

class GUI:
    def __init__(self, x_ball, y_ball, x_basket, y_basket, x_robot, y_robot,
                 a_robot, initial_parameters_step, WIND_X, WIND_Y, scale):
        self.x_ball = x_ball
        self.y_ball = y_ball

        self.x_basket = x_basket
        self.y_basket = y_basket

        self.x_robot = x_robot
        self.y_robot = y_robot
        self.a_robot = a_robot

        self.initial_parameters_step = initial_parameters_step
        
        self.WIND_X = WIND_X
        self.WIND_Y = WIND_Y
        
        self.canvas = np.ones((WIND_Y, WIND_X, 3), np.uint8)

        self.cx = WIND_X // 2
        self.cy = WIND_Y // 2
        self.scale = scale
        
        self.exit_key_pressed = False

    def get_changeable_parameters(self):
        return self.x_ball, self.y_ball, self.x_robot, self.y_robot, self.a_robot

    def _transform_point(self, x, y):
        return (int(self.cx - y * self.scale), int(self.cy - x * self.scale))

    def plot_circle(self, x, y, r, color = (123, 23, 234)):
        center = self._transform_point(x, y)

        cv2.circle(self.canvas, center, r, color, 1)

    def plot_line(self, x1, y1, x2, y2, color = (53, 253, 120)):
        p1 = self._transform_point(x1, y1)
        p2 = self._transform_point(x2, y2)

        cv2.line(self.canvas, p1, p2, color, 2)
    
    def write_text(self, text, x, y, color = (234, 23, 123)):
        origin = self._transform_point(x, y)
        
        image = cv2.putText(self.canvas, text, origin, cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1, cv2.LINE_AA)

    def plot_robot(self, x, y, angle, r = 15, color = (123, 223, 34)):
        self.plot_circle(x, y, r, color)

        center = self._transform_point(x, y)

        cv2.line(self.canvas, center,
                         (int(center[0] - r * 1.5 * math.sin(angle)),
                          int(center[1] - r * 1.5 * math.cos(angle))), color, 2)

    def draw_trajectory(self, state_traj, action_traj, state_name_trajectory):
        self.canvas[:, :, 0] = 20
        self.canvas[:, :, 1] = 130
        self.canvas[:, :, 2] = 57

        #origin and axes
        self.plot_circle(0, 0, 6, (123, 223, 23))
        self.write_text("(0, 0)", 0, 0)
        self.plot_line(0, 0, 2, 0, (123, 223, 23))
        self.write_text("x (2, 0)", 2, 0)
        self.plot_line(0, 0, 0, 2, (123, 223, 23))
        self.write_text("y (0, 2)", 0, 2)

        for i, state in enumerate(state_traj):
            self.plot_robot(state[0], state[1], state[2], color = (state_name_trajectory[i].value * 50, 10, 10))
        
        self.plot_robot(state_traj[0][0], state_traj[0][1], state_traj[0][2], color = (234, 3, 200))
        
        self.plot_circle(self.x_ball, self.y_ball, 20, color = (223, 123, 225))
        self.write_text("ball", self.x_ball, self.y_ball)

        self.plot_circle(self.x_basket, self.y_basket, 20, color = (230, 230, 59))
        self.write_text("basket", self.x_basket, self.y_basket)
    
    def show_canvas(self):
        cv2.imshow("field", self.canvas)
    
    def on_keyboard(self):
        key = cv2.waitKey(8100) & 0xFF
    
        if (key == ord('q')):
            self.exit_key_pressed = True

        #moving ball up and down
        if (key == ord('w')):
            self.x_ball += initial_parameters_step
        if (key == ord('s')):
            self.x_ball -= initial_parameters_step

        #moving ball left and right
        if (key == ord('a')):
            self.y_ball += initial_parameters_step
        if (key == ord('d')):
            self.y_ball -= initial_parameters_step

        #moving robot up and down
        if (key == ord('u')):
            self.x_robot += initial_parameters_step
        if (key == ord('j')):
            self.x_robot -= initial_parameters_step

        #moving robot left and right
        if (key == ord('h')):
            self.y_robot += initial_parameters_step
        if (key == ord('k')):
            self.y_robot -= initial_parameters_step

        #rotating robot clock- and counterclockwise
        if (key == ord('n')):
            self.a_robot += initial_parameters_step
        if (key == ord('m')):
            self.a_robot -= initial_parameters_step
        
    def exit(self):
        return self.exit_key_pressed

#window
x_ball = 1
y_ball = 1

x_basket = 0
y_basket = 1.5

x_robot = 1
y_robot = -1
a_robot = 0.15

FoV = 1.5

initial_parameters_step = 0.1

WIND_X, WIND_Y = 1000, 650
scale = 150

gui = GUI(x_ball, y_ball, x_basket, y_basket, x_robot, y_robot, a_robot,
          initial_parameters_step, WIND_X, WIND_Y, scale)

#simulation
dt = 1
max_simulation_steps = 120

while(not gui.exit()):
    x_ball, y_ball, x_robot, y_robot, a_robot = gui.get_changeable_parameters()

    simulator = Simulator(x_ball, y_ball, x_basket, y_basket, x_robot, y_robot, a_robot, dt, max_simulation_steps, FoV)

    while(simulator.running()):
        simulator.run_simulation_step()
    
    state_traj, action_traj, state_name_trajectory = simulator.get_trajectories()
    gui.draw_trajectory(state_traj, action_traj, state_name_trajectory)

    gui.show_canvas()
    gui.on_keyboard()

cv2.destroyAllWindows()
cv2.waitKey(10)

-1

: 

In [7]:
import numpy as np
import cv2
from enum import Enum
import math

#Generates limited proportional feedback
class Feedback:
    def __init__(self, target_value, tolerance, max_velocity, gain):
        self.target_value = target_value
        self.tolerance    = tolerance
        self.max_velocity = max_velocity
        self.gain         = gain
    
    def generate_feedback(self, value):
        feedback = (value - self.target_value) * self.gain
        
        feedback = np.clip(feedback, -self.max_velocity, self.max_velocity)
        
        return feedback
    
    def stabilized(self, value):
        return abs(value - self.target_value) < self.tolerance

class Approach:
    def __init__(self):
        pass

    def generate_action(self, distance, angle):
        return 0, 0, 0

    def criteria_is_met(self):
        return True

#This approach performs open-loop motion until odometry reaches target for all 3 coordinates
class Approach_by_odometry(Approach):
    def __init__(self, target_x, target_y, target_a,
                       distance_tolerance, angle_tolerance,
                       max_x_velocity, max_y_velocity, max_rotary_velocity,
                       x_velocity, y_velocity, rotary_velocity):
        super().__init__()
        
        self.curr_x = 0
        self.curr_y = 0
        self.curr_a = 0

        self.target_x = target_x
        self.target_y = target_y
        self.target_a = target_a

        self.distance_tolerance = distance_tolerance
        self.angle_tolerance    = angle_tolerance

        self.max_x_velocity      = max_x_velocity
        self.max_y_velocity      = max_y_velocity
        self.max_rotary_velocity = max_rotary_velocity

        self.x_velocity = x_velocity
        self.y_velocity = y_velocity
        self.rotary_velocity = rotary_velocity
        
    def restart(self):
        self.curr_x = 0
        self.curr_y = 0
        self.curr_a = 0

    def generate_action(self, distance, angle):
        vx = self.x_velocity
        vy = self.y_velocity
        omega = self.rotary_velocity

        self.curr_x += vx
        self.curr_y += vy
        self.curr_a += omega
        
        return vx, vy, omega
    
    def criteria_is_met(self, distance, angle):
        #print(abs(self.target_x - self.curr_x),
        #      abs(self.target_y - self.curr_y),
        #      abs(self.target_a       - angle))

        return (abs(self.target_x - self.curr_x) < self.distance_tolerance and
                abs(self.target_y - self.curr_y) < self.distance_tolerance and
                abs(self.target_a       - angle) < self.angle_tolerance)

#This approach works by forward motion and rotation only. No lateral motion is used
class Approach_by_distance_and_heading(Approach):
    def __init__(self, target_distance, target_angle,
                       distance_tolerance, angle_tolerance,
                       max_linear_velocity, max_rotary_velocity,
                       forward_gain, rotary_gain):
        super().__init__()
        
        self.target_distance = target_distance
        self.target_angle    = target_angle
        
        self.distance_tolerance = distance_tolerance
        self.angle_tolerance    = angle_tolerance
        
        self.max_linear_velocity = max_linear_velocity
        self.max_rotary_velocity = max_rotary_velocity
        
        self.forward_gain = forward_gain
        self.rotary_gain  = rotary_gain
        
        self.distance_feedback = Feedback(self.target_distance, self.distance_tolerance,
                                          self.max_linear_velocity, self.forward_gain)

        self.angle_feedback = Feedback(self.target_angle, self.angle_tolerance,
                                          self.max_rotary_velocity, self.rotary_gain)
        
    def generate_action(self, distance, angle):
        vx = self.distance_feedback.generate_feedback(distance)
        vy = 0
        omega = self.angle_feedback.generate_feedback(angle)
        
        return vx, vy, omega
    
    def criteria_is_met(self, distance, angle):
        return (abs(self.target_distance - distance) < self.distance_tolerance and
                abs(self.target_angle    - angle)    < self.angle_tolerance)

#This approach parforms saggital, lateral and rotational P compensation independently.
class Approach_by_xytheta(Approach):
    def __init__(self, target_x, target_y, target_a,
                       distance_tolerance, angle_tolerance,
                       max_x_velocity, max_y_velocity, max_rotary_velocity,
                       forward_gain, lateral_gain, rotary_gain):
        super().__init__()
        
        self.target_x = target_x
        self.target_y = target_y
        self.target_a = target_a
        
        self.distance_tolerance = distance_tolerance
        self.angle_tolerance    = angle_tolerance
        
        self.max_x_velocity = max_x_velocity
        self.max_y_velocity = max_y_velocity
        self.max_rotary_velocity = max_rotary_velocity
        
        self.forward_gain = forward_gain
        self.lateral_gain = lateral_gain
        self.rotary_gain  = rotary_gain
        
        self.x_feedback = Feedback(self.target_x, self.distance_tolerance,
                                   self.max_x_velocity, self.forward_gain)

        self.y_feedback = Feedback(self.target_y, self.distance_tolerance,
                                   self.max_y_velocity, self.lateral_gain)

        self.angle_feedback = Feedback(self.target_a, self.angle_tolerance,
                                          self.max_rotary_velocity, self.rotary_gain)
        
    def generate_action(self, x, y, angle):
        vx = self.x_feedback.generate_feedback(x)
        vy = self.y_feedback.generate_feedback(y)
        omega = self.angle_feedback.generate_feedback(angle)
        
        return vx, vy, omega
    
    def criteria_is_met(self, x, y, angle):
        #return (abs(self.target_x - distance) < self.distance_tolerance and
        #        abs(self.target_angle    - angle)    < self.angle_tolerance)
        
        return self.x_feedback.stabilized(x) and self.y_feedback.stabilized(y) and self.angle_feedback.stabilized(angle)

class States(Enum):
    INIT = 0
    FORWARD = 1
    BACKWARD = 2
    EXIT = 3

class Actions(Enum):
    WALKING = 0
    TAKING_BALL = 1
    THROWING_BALL = 2

class Sprint_controller:
    def __init__(self, switch_distance, finish_distance, distance_tolerance, angle_tolerance, max_x_velocity,
                 max_y_velocity, max_rotary_velocity, forward_gain, lateral_gain, rotary_gain):
        self.state = States.INIT
        self.prev_state = self.state
        self.prev_action = (0, 0, 0)
        
        self.forward_approach = Approach_by_xytheta(switch_distance, 0, 0,
            distance_tolerance, angle_tolerance, max_x_velocity, max_y_velocity,
            max_rotary_velocity, forward_gain, lateral_gain, rotary_gain)

        self.backward_approach = Approach_by_xytheta(finish_distance, 0, 0,
            distance_tolerance, angle_tolerance, max_x_velocity, max_y_velocity,
            max_rotary_velocity, forward_gain, lateral_gain, rotary_gain)
    
    def generate_action(self, observation):
        #separate into two functions (?): handling FSM and generating action itself

        (x_aruco_local, y_aruco_local, theta_aruco_local, aruco_visible) = observation

        self.prev_state = self.state

        if (self.state == States.INIT):
            self.state = States.FORWARD
        
        #defaulting to straight walk if aruco is missing
        if (not aruco_visible):
            return self.prev_action, Actions.WALKING, self.state
        
        if (self.state == States.FORWARD):
            switch_approached = self.forward_approach.criteria_is_met(x_aruco_local, y_aruco_local, theta_aruco_local)

            if (switch_approached):
                self.state = States.BACKWARD

        # if (self.state == States.BACKWARD):
        #     finish_approached = self.backward_approach.criteria_is_met(x_aruco_local, y_aruco_local, yaw_aruco_local)

        #     if (finish_approached):
        #         self.state = States.EXIT

        #generating action
        action_name = Actions.WALKING
        vx = 0
        vy = 0
        omega = 0

        if (self.state == States.FORWARD):
            vx, vy, omega = self.forward_approach.generate_action(x_aruco_local, y_aruco_local, theta_aruco_local)
        
        if (self.state == States.BACKWARD):
            vx, vy, omega = self.backward_approach.generate_action(x_aruco_local, y_aruco_local, theta_aruco_local)
        
        #if state = EXIT, walking is performed with zero speed

        self.prev_action = (vx, vy, omega)
        
        #print(self.prev_action)

        return (vx, vy, omega), action_name, self.state
    
    def get_state(self):
        return self.state

class Robot:
    def __init__(self, x0, y0, a0, dt):
        self.x = x0
        self.y = y0
        self.a = a0 #alpha
        self.dt = dt
    
    def walk(self, velocities):
        vx, vy, omega = velocities

        self.x += vx * self.dt * math.cos(self.a) - vy * self.dt * math.sin(self.a) + np.random.randn() * 0.03
        self.y += vx * self.dt * math.sin(self.a) + vy * self.dt * math.cos(self.a) + np.random.randn() * 0.03
        self.a += omega * dt

    def get_state(self):
        return (self.x, self.y, self.a)
        
class Object:
    def __init__(self, x_obj, y_obj):
        self.x_obj = x_obj
        self.y_obj = y_obj
        
    def get_observation(self, x_robot, y_robot, a_robot, FoV):
        R = np.array([[np.cos(a_robot), np.sin(a_robot)],
                      [-np.sin(a_robot), np.cos(a_robot)]])
        
        X_object_glob = np.array([[self.x_obj], [self.y_obj]])
        X_robot_glob  = np.array([[x_robot], [y_robot]])

        translated_rotated = R @ (X_object_glob - X_robot_glob)

        x_loc, y_loc = translated_rotated[:, 0]

        angle = math.atan2(y_loc, x_loc)

        object_visible = abs(angle) < FoV / 2

        return x_loc, y_loc, object_visible

class Object_with_orientation(Object):
    def __init__(self, x_obj, y_obj, theta_obj):
        super().__init__(x_obj, y_obj)
        
        self.theta_obj = theta_obj
        
    def get_observation(self, x_robot, y_robot, a_robot, FoV):
        R = np.array([[np.cos(a_robot), np.sin(a_robot)],
                      [-np.sin(a_robot), np.cos(a_robot)]])
        
        X_object_glob = np.array([[self.x_obj], [self.y_obj]])
        X_robot_glob  = np.array([[x_robot], [y_robot]])

        translated_rotated = R @ (X_object_glob - X_robot_glob)

        x_loc, y_loc = translated_rotated[:, 0]

        angle = math.atan2(y_loc, x_loc)

        object_visible = abs(angle) < FoV / 2
        
        if (np.random.randint(10) < 3):
            object_visible = False

        return x_loc, y_loc, self.theta_obj - a_robot, object_visible

class World:
    def __init__(self, x_aruco, y_aruco, theta_aruco):
        self.x_aruco = x_aruco
        self.y_aruco = y_aruco
        self.theta_aruco = theta_aruco
        
        self.aruco = Object_with_orientation(self.x_aruco, self.y_aruco, self.theta_aruco)
    
    def get_observation(self, robot_state, FoV):
        x_robot, y_robot, a_robot = robot_state

        x_aruco_local, y_aruco_local, yaw_aruco_local, aruco_visible = self.aruco.get_observation(x_robot, y_robot, a_robot, FoV)
        
        return x_aruco_local, y_aruco_local, yaw_aruco_local, aruco_visible

class Simulator:
    def __init__(self, x_aruco, y_aruco, theta_aruco, x_robot, y_robot, a_robot,
                 dt, max_simulation_steps, FoV,
                 
                 ):
        self.x_aruco = x_aruco
        self.y_aruco = y_aruco
        self.theta_aruco = theta_aruco

        self.x_robot = x_robot
        self.y_robot = y_robot
        self.a_robot = a_robot

        self.dt = dt
        self.max_simulation_steps = max_simulation_steps

        self.FoV = FoV

        self.robot = Robot(self.x_robot, self.y_robot, self.a_robot, self.dt)
        
        switch_distance = 0.3
        finish_distance = 4
        distance_tolerance = 0.1
        angle_tolerance = 0.2
        max_x_velocity = 0.1
        max_y_velocity = 0.05
        max_rotary_velocity = 0.15
        forward_gain = 1
        lateral_gain = 1
        rotary_gain = 1

        self.controller = Sprint_controller(switch_distance, finish_distance, distance_tolerance, angle_tolerance, max_x_velocity,
                 max_y_velocity, max_rotary_velocity, forward_gain, lateral_gain, rotary_gain)
        
        self.world = World(self.x_aruco, self.y_aruco, self.theta_aruco)

        self.max_simulation_steps = max_simulation_steps
        self.iter_num = 0
        
        self.state_trajectory = []
        self.state_name_trajectory = []
        self.action_trajectory = []
    
    def run_simulation_step(self):
        robot_state = self.robot.get_state()
        
        observation = self.world.get_observation(robot_state, self.FoV)
        
        #(vx, vy, omega), action_name, self.state
        action, action_name, state_name = self.controller.generate_action(observation)

        self.robot.walk(action)
        
        self.state_trajectory.append(robot_state)
        self.action_trajectory.append(action)
        self.state_name_trajectory.append(state_name)

        self.iter_num += 1
        
    def get_trajectories(self):
        return self.state_trajectory, self.action_trajectory, self.state_name_trajectory
    
    def running(self):
        return (self.controller.get_state() != States.EXIT and self.iter_num < self.max_simulation_steps)

class GUI:
    def __init__(self, x_aruco, y_aruco, theta_aruco, x_robot, y_robot,
                 a_robot, initial_parameters_step, WIND_X, WIND_Y, scale):
        self.x_aruco = x_aruco
        self.y_aruco = y_aruco
        self.theta_aruco = theta_aruco

        self.x_robot = x_robot
        self.y_robot = y_robot
        self.a_robot = a_robot

        self.initial_parameters_step = initial_parameters_step
        
        self.WIND_X = WIND_X
        self.WIND_Y = WIND_Y
        
        self.canvas = np.ones((WIND_Y, WIND_X, 3), np.uint8)

        self.cx = WIND_X * 2 // 3
        self.cy = WIND_Y * 4 // 5
        self.scale = scale
        
        self.exit_key_pressed = False

    def get_changeable_parameters(self):
        return self.x_aruco, self.y_aruco, self.theta_aruco, self.x_robot, self.y_robot, self.a_robot

    def _transform_point(self, x, y):
        return (int(self.cx - y * self.scale), int(self.cy - x * self.scale))

    def plot_circle(self, x, y, r, color = (123, 23, 234)):
        center = self._transform_point(x, y)

        cv2.circle(self.canvas, center, r, color, 1)

    def plot_line(self, x1, y1, x2, y2, color = (53, 253, 120)):
        p1 = self._transform_point(x1, y1)
        p2 = self._transform_point(x2, y2)

        cv2.line(self.canvas, p1, p2, color, 2)
    
    def write_text(self, text, x, y, color = (234, 23, 123)):
        origin = self._transform_point(x, y)
        
        image = cv2.putText(self.canvas, text, origin, cv2.FONT_HERSHEY_SIMPLEX, 1.6, color, 2, cv2.LINE_AA)

    def plot_robot(self, x, y, angle, r = 35, color = (123, 223, 34)):
        self.plot_circle(x, y, r, color)

        center = self._transform_point(x, y)

        cv2.line(self.canvas, center,
                         (int(center[0] - r * 1.5 * math.sin(angle)),
                          int(center[1] - r * 1.5 * math.cos(angle))), color, 2)

    def draw_trajectory(self, state_traj, action_traj, state_name_trajectory):
        self.canvas[:, :, 0] = 20
        self.canvas[:, :, 1] = 130
        self.canvas[:, :, 2] = 57

        #origin and axes
        self.plot_circle(0, 0, 6, (123, 223, 23))
        self.write_text("(0, 0)", 0, 0)
        self.plot_line(0, 0, 3, 0, (123, 223, 23))
        self.write_text("x (3, 0)", 3, 0)
        self.plot_line(0, 0, 0, 1, (123, 223, 23))
        self.write_text("y (0, 1)", 0, 1)

        for i, state in enumerate(state_traj):
            #self.plot_robot(state[0], state[1], state[2], color = (state_name_trajectory[i].value * 50, 10, 10))
            self.plot_robot(state[0], state[1], state[2], color = (i * 1, i * 0, i * 2))
        
        self.plot_robot(state_traj[0][0], state_traj[0][1], state_traj[0][2], color = (234, 3, 200))
        
        self.plot_circle(self.x_aruco, self.y_aruco, 50, color = (223, 123, 225))
        self.plot_line(self.x_aruco, self.y_aruco, self.x_aruco + 0.1 * np.cos(self.theta_aruco),
                                                   self.y_aruco + 0.1 * np.sin(self.theta_aruco), (123, 223, 223))
        
        self.write_text("aruco", self.x_aruco + 50, self.y_aruco)
    
    def show_canvas(self):
        cv2.imshow("field", self.canvas)
    
    def on_keyboard(self):
        key = cv2.waitKey(3100) & 0xFF
    
        if (key == ord('q')):
            self.exit_key_pressed = True

        #moving ball up and down
        if (key == ord('w')):
            self.x_aruco += initial_parameters_step
        if (key == ord('s')):
            self.x_aruco -= initial_parameters_step

        #moving ball left and right
        if (key == ord('a')):
            self.y_aruco += initial_parameters_step
        if (key == ord('d')):
            self.y_aruco -= initial_parameters_step

        #moving robot up and down
        if (key == ord('u')):
            self.x_robot += initial_parameters_step
        if (key == ord('j')):
            self.x_robot -= initial_parameters_step

        #moving robot left and right
        if (key == ord('h')):
            self.y_robot += initial_parameters_step
        if (key == ord('k')):
            self.y_robot -= initial_parameters_step

        #rotating robot clock- and counterclockwise
        if (key == ord('n')):
            self.a_robot += initial_parameters_step
        if (key == ord('m')):
            self.a_robot -= initial_parameters_step
        
    def exit(self):
        return self.exit_key_pressed

#window
x_aruco = 3.5
y_aruco = 0.5
theta_aruco = 0.0

x_robot = -0.1
y_robot = 0.4
a_robot = 0.15

FoV = 1.5

initial_parameters_step = 0.1

WIND_X, WIND_Y = 1000, 1900
scale = 400

gui = GUI(x_aruco, y_aruco, theta_aruco, x_robot, y_robot, a_robot,
          initial_parameters_step, WIND_X, WIND_Y, scale)

#simulation
dt = 1
max_simulation_steps = 100

while(not gui.exit()):
    x_aruco, y_aruco, theta_aruco, x_robot, y_robot, a_robot = gui.get_changeable_parameters()

    simulator = Simulator(x_aruco, y_aruco, theta_aruco, x_robot, y_robot, a_robot, dt, max_simulation_steps, FoV)

    while(simulator.running()):
        simulator.run_simulation_step()
    
    state_traj, action_traj, state_name_trajectory = simulator.get_trajectories()
    gui.draw_trajectory(state_traj, action_traj, state_name_trajectory)

    gui.show_canvas()
    gui.on_keyboard()

cv2.destroyAllWindows()
cv2.waitKey(10)

-1